In [1]:
import numpy as np

In [2]:
# helix (axis along x) that starts at top and ends at bottom
n_nodes = 500
n_edges = n_nodes - 1 +2  # +2 for start and end nodes
nodes, edges = [], []

R_n = 0.034
L = 0.5
n_turns = 10          # "turn count" used for pairing


theta_start = 0.0
# theta_end = (2*n_turns) * np.pi   # ends at bottom
theta_end = (2*n_turns - 1) * np.pi   # ends at top
del_theta = (theta_end - theta_start) / (n_nodes - 1)

# choose x mapping so that x goes from 0 to L over the whole sampled helix
# i.e., L corresponds to theta_end
x_per_rad = L / (theta_end - theta_start)

top_nodes_idx = []
bottom_nodes_idx = []

for c in range(n_nodes):
    theta = theta_start + c * del_theta

    node_y = R_n * np.cos(theta)
    node_z = R_n * np.sin(theta)
    node_x = x_per_rad * (theta - theta_start)

    nodes.append([node_x, node_y, node_z])

    # exact index sets (no floating checks)
    # top targets: 0, 2π, 4π, ..., 2π*(n_turns-1)
    # bottom targets: π, 3π, ..., (2n_turns-1)π
for k in range(n_turns):
    target_top = 2*np.pi*k
    idx_top = int(round((target_top - theta_start)/del_theta))
    top_nodes_idx.append(np.clip(idx_top, 0, n_nodes-1))

    target_bot = np.pi + 2*np.pi*k
    idx_bot = int(round((target_bot - theta_start)/del_theta))
    bottom_nodes_idx.append(np.clip(idx_bot, 0, n_nodes-1))

nodes = np.asarray(nodes, dtype=float)

print("top_nodes_idx:", top_nodes_idx)
print("bottom_nodes_idx:", bottom_nodes_idx)

top_nodes = nodes[top_nodes_idx,:]
bottom_nodes = nodes[bottom_nodes_idx,:]

start_node = np.array([0.0, 0.0, 0.0])
end_node = np.array([L, 0.0, 0.0])
nodes = np.vstack((start_node, nodes, end_node))

top_nodes_idx: [np.int64(0), np.int64(53), np.int64(105), np.int64(158), np.int64(210), np.int64(263), np.int64(315), np.int64(368), np.int64(420), np.int64(473)]
bottom_nodes_idx: [np.int64(26), np.int64(79), np.int64(131), np.int64(184), np.int64(236), np.int64(289), np.int64(341), np.int64(394), np.int64(446), np.int64(499)]


In [3]:
# make edges (1-based)
for c in range(n_edges):
    edges.append([c+1, c+2])

In [4]:
from pathlib import Path

def write_input_txt(filepath, V, E):
    """
    Write ribbon mesh to custom text format.

    Parameters
    ----------
    filepath : str or Path
        Full path including filename
    V : (N,3) array
        Vertex positions
    E : (E,2) array
        Edge indices
    """
    filepath = Path(filepath)  # converts str → Path safely
    filepath.parent.mkdir(parents=True, exist_ok=True)  # create dirs if needed

    with filepath.open("w") as f:
        #f.write("*Nodes\n")
        for v in V:
            f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")

        #f.write("\n*Edges\n")
        #for edge in E:
        #    f.write(f"{edge[0]}, {edge[1]}\n")

In [5]:
write_input_txt("slinky.txt", nodes, edges)